## Native workbook charts
Run in the dashboards folder. Build separate native views from the same CSV as the standalone Python dashboard; keep the original detail sheets.

In [ ]:
from pathlib import Path
import base64
import csv
import gzip
import json
from copy import deepcopy
import xml.etree.ElementTree as ET

here = Path.cwd()
results = here.parent / "results"
with (results/"strategy_cells.csv").open(encoding="utf-8", newline="") as src:
    cells = list(csv.DictReader(src))
views = list(dict.fromkeys(["Deployment routes"]+[r["view"] for r in cells]))
desc = {"Printed curve":"Nonland library slots", "Card types":"Slots in the full deck","Creature types":"Printed subtypes leading at least one deck"}
desc.update({"Commander identity":"Color permitted by commander(s): 1 yes, 0 no","Printed card colors":"Share of nonland slots (including commander)","Casting-cost emphasis":"Percent of colored casting symbols","Creature composition":"Exclusive creature-card slots by other types","Human-adjusted tribes":"Creature slots; omit Human only when accompanied by another subtype"})
for view in views: desc.setdefault(view,"Distinct cards with this capability")
def readj(path):
    return json.loads(path.read_text(encoding="utf-8"))
def writej(path,obj):
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(obj,indent=2,ensure_ascii=False)+"\n",encoding="utf-8")


In [ ]:
path = here/"tableau/gameplay.twb"
tree = ET.parse(path)
root=tree.getroot()
add=ET.SubElement
sources,sheets,boards=[root.find(k) for k in ("datasources","worksheets","dashboards")]
for parent in (sources,sheets,boards):
    for child in list(parent):
        if child.get("name")=="strategy" or child.get("name","").startswith("Compare | "):
            parent.remove(child)
source=add(sources,"datasource",{"caption":"Deck capabilities","inline":"true","name":"strategy","version":"10.5"})
conn=add(source,"connection",{"class":"textscan","directory":"../../results","filename":"strategy_cells.csv","password":"","server":""})
cols=add(add(conn,"relation",{"name":"strategy_cells","table":"[strategy_cells#csv]","type":"table"}),"columns",{"header":"yes","separator":",","charset":"UTF-8"})
names=list(cells[0])
def fld(name):return "[strategy]."+instance[name]
instance={}
for i,n in enumerate(names):
    numeric=n in ("count","position","slot_share")
    typ="real" if n=="slot_share" else "integer" if numeric else "string"
    add(cols,"column",{"datatype":typ,"name":n,"ordinal":str(i)})
    add(source,"column",{"datatype":typ,"name":f"[{n}]","role":"measure" if numeric else "dimension","type":"quantitative" if numeric else "nominal"})
    instance[n]=f"[sum:{n}:qk]" if numeric else f"[none:{n}:nk]"
for i,title in enumerate(views):
    name="Compare | "+title
    w=ET.Element("worksheet",name=name+" marks")
    sheets.insert(i,w)
    heading=add(add(add(w,"layout-options"),"title"),"formatted-text")
    add(heading,"run",bold="true").text=title
    add(heading,"run").text="  |  "+desc[title]
    tab=add(w,"table");v=add(tab,"view")
    add(add(v,"datasources"),"datasource",{"caption":"Deck capabilities","name":"strategy"})
    deps=add(v,"datasource-dependencies",datasource="strategy")
    for c in source.findall("column"): deps.append(deepcopy(c))
    for n in names:
        numeric=n in ("count","position","slot_share")
        add(deps,"column-instance",{"column":f"[{n}]","derivation":"Sum" if numeric else "None","name":instance[n],"pivot":"key","type":"quantitative" if numeric else "nominal"})
    filt=add(v,"filter",{"class":"categorical","column":fld("view")})
    add(filt,"groupfilter",{"function":"member","level":instance["view"],"member":json.dumps(title)})
    add(v,"aggregation",value="true")
    add(tab,"style")
    pane=add(add(tab,"panes"),"pane")
    add(add(pane,"view"),"breakdown",value="auto")
    add(pane,"mark",{"class":"Square"})
    enc=add(pane,"encodings")
    for k,n in [("color","count"),("text","count"),("tooltip","cards")]:add(enc,k,column=fld(n))
    tip=add(add(pane,"customized-tooltip"),"formatted-text")
    add(tip,"run",bold="true").text=fld("deck")+" / "+fld("category")+"\n"
    add(tip,"run").text=fld("count")+"  |  "+fld("cards")
    add(tab,"rows").text=fld("field_label")
    add(tab,"cols").text=fld("category")
    if title=="Printed card colors":
        pane.find("mark").set("class","Bar"); enc.clear()
        add(enc,"color",column=fld("category"));add(enc,"tooltip",column=fld("count"));add(enc,"tooltip",column=fld("cards"))
        tab.find("cols").text=fld("slot_share")
        tab.find("style").clear()
    d=ET.Element("dashboard",name=name)
    boards.insert(i,d);add(d,"style")
    add(d,"size",{"minwidth":"1150","maxwidth":"1650","minheight":"720","maxheight":"920","sizing-mode":"range"})
    add(add(d,"zones"),"zone",{"id":"1","name":w.get("name"),"x":"0","y":"0","w":"100000","h":"100000"})
# One player and one saved reference, with native Tableau quick filters.
for parent in (sources,sheets,boards):
    for child in list(parent):
        if child.get("name") in ("pairs","Head-to-head marks","Head-to-head"): parent.remove(child)
with (results/"strategy_pairs.csv").open(encoding="utf-8",newline="") as source_file:
    pair_headers=next(csv.reader(source_file))
pair_source=deepcopy(source);pair_source.set("name","pairs");pair_source.set("caption","Player and public reference")
sources.append(pair_source)
connection=pair_source.find("connection")
connection.set("filename","strategy_pairs.csv")
relation=connection.find("relation");relation.set("name","strategy_pairs");relation.set("table","[strategy_pairs#csv]")
columns=relation.find("columns")
for node in list(columns):columns.remove(node)
for node in pair_source.findall("column"):pair_source.remove(node)
sheet=deepcopy(sheets[0]);sheet.set("name","Head-to-head marks");sheets.insert(0,sheet)
sheet.find(".//formatted-text").clear()
add(sheet.find(".//formatted-text"),"run",bold="true").text="Your deck vs. public reference | card count"
sheet_view=sheet.find("table/view")
sheet_view.find("datasources/datasource").attrib.update(name="pairs",caption="Player and public reference")
deps=sheet_view.find("datasource-dependencies");deps.clear();deps.set("datasource","pairs")
for i,k in enumerate(pair_headers):
    numeric=k in ("count","position");ty="integer" if numeric else "string"
    add(columns,"column",{"name":k,"datatype":ty,"ordinal":str(i)})
    col=add(pair_source,"column",{"name":f"[{k}]","datatype":ty,"role":"measure" if numeric else "dimension","type":"quantitative" if numeric else "nominal"})
    deps.append(deepcopy(col))
for k in pair_headers:
    numeric=k in ("count","position")
    add(deps,"column-instance",{"column":f"[{k}]","derivation":"Sum" if numeric else "None","name":f"[sum:{k}:qk]" if numeric else f"[none:{k}:nk]","pivot":"key","type":"quantitative" if numeric else "nominal"})
for f in sheet_view.findall("filter"):sheet_view.remove(f)
default_reference=next(r["field_label"] for r in cells if r["group"]!="Your deck")
def pair_field(n):
    return "[pairs]."+("[sum:count:qk]" if n=="count" else f"[none:{n}:nk]")
for k,value in [("reference_choice",default_reference),("view","Deployment routes")]:
    f=ET.Element("filter",{"class":"categorical","column":pair_field(k)})
    add(f,"groupfilter",{"function":"member","level":f"[none:{k}:nk]","member":json.dumps(value)})
    sheet_view.insert(list(sheet_view).index(sheet_view.find("aggregation")),f)
sheet.find("table/style").clear()
pane=sheet.find(".//panes/pane");pane.find("mark").set("class","Bar")
enc=pane.find("encodings");enc.clear()
for k,col in [("color","side"),("text","count"),("tooltip","cards"),("lod","deck")]:add(enc,k,column=pair_field(col))
form=pane.find("customized-tooltip/formatted-text");form.clear()
add(form,"run").text=pair_field("deck")+" | "+pair_field("category")+"\n"+pair_field("cards")
sheet.find("table/rows").text="("+pair_field("category")+" / "+pair_field("side")+")"
sheet.find("table/cols").text=pair_field("count")
board=deepcopy(boards[0]);board.set("name","Head-to-head");boards.insert(0,board)
zones=board.find("zones");plot=zones.find("zone");plot.attrib.update(name="Head-to-head marks",y="13000",h="87000")
for i,k in enumerate(["reference_choice","view"]):
    add(zones,"zone",{"id":str(2+i),"name":"Head-to-head marks","type-v2":"filter","param":pair_field(k),"mode":"checkdropdown","x":str(i*50000),"y":"0","w":"50000","h":"12500"})

ET.indent(tree,space="  ")
tree.write(path,encoding="utf-8",xml_declaration=True)


In [ ]:
base=here/"power-bi"
model_path=base/"Comparison.SemanticModel/model.bim"
pages=base/"Comparison.Report/definition/pages"
model=readj(model_path)
m=model["model"]
def snapshot(csvname):
    return base64.b64encode(gzip.compress((results/csvname).read_bytes(),mtime=0)).decode("ascii")
def read_csv(file,param):
    return ["let",f'  Source = if {param} = null or {param} = "" then Binary.Decompress(Binary.FromText("{snapshot(file)}",BinaryEncoding.Base64),Compression.GZip) else File.Contents({param}),',
            '  Rows = Csv.Document(Source,[Delimiter=",",Encoding=65001,QuoteStyle=QuoteStyle.Csv]),',
            '  Headers = Table.PromoteHeaders(Rows,[PromoteAllScalars=true])','in Headers']
overlap = next(t for t in m["tables"] if t["name"]=="Comparisons")
overlap_csv = read_csv("deck_comparisons.csv","ResultsFile")[:-1]
overlap_csv[-1] += ","
overlap_csv += ['  Types = Table.TransformColumnTypes(Headers, {{"shared_card_names", Int64.Type}})', "in Types"]
overlap["partitions"][0]["source"]["expression"] = overlap_csv
expressions={e["name"]:e for e in m["expressions"]}
expressions["GameplayData"]["expression"]=read_csv("gameplay_dashboard.csv","GameplayFile")
m["expressions"]=[e for e in m["expressions"] if e["name"] not in ("StrategyFile","StrategyData")]+[
    {**expressions["GameplayFile"],"name":"StrategyFile"},
    {"name":"StrategyData","kind":"m","expression":read_csv("strategy_cells.csv","StrategyFile")}]
m["tables"]=[t for t in m["tables"] if not t["name"].startswith("Compare | ")]
chooser="Public reference"
m["tables"]=[table for table in m["tables"] if table["name"]!=chooser]
m["tables"].append({"name":chooser,"columns":[{"name":"deck","dataType":"string","sourceColumn":"deck","summarizeBy":"none","sortByColumn":"deck_order"},{"name":"deck_order","dataType":"int64","sourceColumn":"deck_order","summarizeBy":"none"}],
    "partitions":[{"name":chooser,"mode":"import","source":{"type":"m","expression":'Table.TransformColumnTypes(Table.Distinct(Table.SelectColumns(Table.SelectRows(StrategyData, each [group] <> "Your deck"), {"deck","deck_order"})), {{"deck_order",Int64.Type}})'}}]})
default_ref=next(r["deck"] for r in cells if r["group"]!="Your deck").replace('"','""')

page_format=readj(pages/"ba68a5b433de4872a101/page.json")
bar_format=readj(next((pages/"ba68a5b433de4872a101").glob("visuals/*/visual.json")))
index=readj(pages/"pages.json")
ids=[]
def literal(v):return {"expr":{"Literal":{"Value":v}}}
def color(v):return {"solid":{"color":literal("'"+v+"'")}}
for i,view in enumerate(views):
    table="Compare | "+view
    fields=[]
    for n in cells[0]:
        if n=="view":continue
        column={"name":n,"dataType":"int64" if n in ("count","position","deck_order","group_order") else "string","sourceColumn":n,"summarizeBy":"none"}
        if n in ("deck","field_label"):column["sortByColumn"]="deck_order"
        if n=="group":column["sortByColumn"]="group_order"
        if n=="category":column["sortByColumn"]="position"
        fields.append(column)
    measures=[{"name":"Cell count","expression":f'SELECTEDVALUE(\'{table}\'[count])',"formatString":"0"},
              {"name":"Group mean","expression":f'AVERAGE(\'{table}\'[count])',"formatString":"0.0"},
              {"name":"Cell color","expression":f'VAR peak = MAXX(ALL(\'{table}\'), \'{table}\'[count]) VAR ratio = DIVIDE([Cell count], peak, 0) RETURN SWITCH(TRUE(), ISBLANK([Cell count]), "#FFFFFF", ratio < 0.08, "#F4F6EC", ratio < 0.28, "#D1E8C7", ratio < 0.50, "#99D2BC", ratio < 0.75, "#67BBBE", "#409AB0")'},
              {"name":"Card names","expression":f'SELECTEDVALUE(\'{table}\'[cards], "Select a deck and category to see the cards.")'}]
    m["tables"].append({"name":table,"columns":fields,"measures":measures,"partitions":[{"name":table,"mode":"import","source":{"type":"m","expression":[
        f'let Rows = Table.SelectRows(StrategyData, each [view] = "{view}"),',
        '    Clean = Table.ReplaceValue(Rows,"",null,Replacer.ReplaceValue,{"count"}),',
        '    Types = Table.TransformColumnTypes(Clean,{{"count",Int64.Type},{"position",Int64.Type},{"deck_order",Int64.Type},{"group_order",Int64.Type}}) in Types']}}]})
    def proj(n,measure=False):
        kind="Measure" if measure else "Column"
        return {"field":{kind:{"Expression":{"SourceRef":{"Entity":table}},"Property":n}},"queryRef":table+"."+n}
    def state(**kw):
        return {k:{"projections":[proj(n,is_measure) for n,is_measure in vals]} for k,vals in kw.items()}
    pid=f"a928f7873dc262f2{i:04x}";ids.append(pid)
    page={**page_format,"name":pid,"displayName":view,"height":930,"width":1500}
    writej(pages/pid/"page.json",page)
    def visual(j,kind,where,qs,title,objs=None):
        vid=f"ca81d5706aa17{i:03x}{j:04x}"
        x,y,w,h=where
        heading={"title":[{"properties":{"show":literal("true"),"text":literal("'"+title+"'")}}]}
        obj={"$schema":bar_format["$schema"],"name":vid,"position":{"x":x,"y":y,"width":w,"height":h,"z":j,"tabOrder":j},
             "visual":{"visualType":kind,"query":{"queryState":qs},"visualContainerObjects":heading}}
        if objs:obj["visual"]["objects"]=objs
        writej(pages/pid/"visuals"/vid/"visual.json",obj)
    shade={"solid":{"color":{"expr":proj("Cell color",True)["field"]}}}
    objs={"values":[{"selector":{"metadata":table+".Cell count"},"properties":{"backColor":shade}}],
          "subTotals":[{"properties":{"rowSubtotals":literal("false"),"columnSubtotals":literal("false")}}]}
    visual(1,"pivotTable",(12,10,1476,545),state(Rows=[("field_label",False)],Columns=[("category",False)],Values=[("Cell count",True)]),view+" | "+desc[view],objs)
    if view=="Printed card colors":
        visual(1,"hundredPercentStackedBarChart",(12,10,1476,545),state(Category=[("field_label",False)],
               Series=[("category",False)],Y=[("Cell count",True)]),"Nonland color composition | share of nonland slots")
    # A disconnected choice filters only the reference series. All twenty matrix rows remain.
    ref_pick=f'SELECTEDVALUE(\'Public reference\'[deck], "{default_ref}")'
    measures.extend([{"name":"Your deck","expression":f'CALCULATE(MAX(\'{table}\'[count]), \'{table}\'[group] = "Your deck")',"formatString":"0"},
        {"name":"Selected reference","expression":f'VAR selectedDeck = {ref_pick} RETURN CALCULATE(MAX(\'{table}\'[count]), \'{table}\'[deck] = selectedDeck)',"formatString":"0"},
        {"name":"Pair title","expression":f'VAR own = CALCULATE(SELECTEDVALUE(\'{table}\'[deck]), \'{table}\'[group] = "Your deck") RETURN own & " vs. " & {ref_pick}'}])
    visual(2,"clusteredBarChart",(12,560,1015,355),state(Category=[("category",False)],Y=[("Your deck",True),("Selected reference",True)]),"Your deck vs. selected public reference")
    visual_path=next(p for p in (pages/pid/"visuals").glob("*/*") if p.parent.name.endswith("0002"))
    chart=readj(visual_path)
    chart["visual"]["visualContainerObjects"]["title"][0]["properties"]["text"]={"expr":proj("Pair title",True)["field"]}
    writej(visual_path,chart)
    visual(4,"slicer",(1035,560,450,145),{"Values":{"projections":[{"field":{"Column":{"Expression":{"SourceRef":{"Entity":chooser}},"Property":"deck"}},"queryRef":chooser+".deck","active":True}]}},"Public reference",{"selection":[{"properties":{"singleSelect":literal("true")}}]})
    visual(3,"multiRowCard",(1035,710,450,205),state(Values=[("Card names",True)]),"Cards in the selected cell")
writej(model_path,model)
index["pageOrder"]=ids+[p for p in index["pageOrder"] if p not in ids]
index["activePageName"]=ids[0]
writej(pages/"pages.json",index)


In [ ]:
detail_file="review_details.csv"
with (results/detail_file).open(encoding="utf-8",newline="") as src: detail_fields=next(csv.reader(src))
pages_to_fields={"Deck routes":["heading","detail"],"Card evidence":["heading","cost","timing","conditions","detail","url"],
                 "Bracket means":["deck","heading","detail","conditions"],"Definitions":["heading","detail"]}
player_name=next(r["deck"] for r in cells if r["group"]=="Your deck")
# Each workbook reads the common detail facts, not a copy of another dashboard's rendered charts.
for parent in (sources,sheets,boards):
    for child in list(parent):
        if child.get("name")=="review" or child.get("name","").startswith("Evidence | "):parent.remove(child)
review_source=deepcopy(pair_source);review_source.attrib.update(name="review",caption="Gameplay evidence");sources.append(review_source)
con=review_source.find("connection");con.set("filename",detail_file)
rel=con.find("relation");rel.attrib.update(name="review_details",table="[review_details#csv]")
cols=rel.find("columns")
for n in list(cols):cols.remove(n)
for n in review_source.findall("column"):review_source.remove(n)
for i,n in enumerate(detail_fields):
    ty="integer" if n=="deck_order" else "string"
    add(cols,"column",name=n,datatype=ty,ordinal=str(i))
    add(review_source,"column",name=f"[{n}]",datatype=ty,role="dimension",type="nominal")
def review_field(n):return "[review].[none:"+n+":nk]"
for title in pages_to_fields:
    name="Evidence | "+title
    w=ET.Element("worksheet",name=name+" marks");sheets.append(w)
    t=add(w,"table");v=add(t,"view");add(add(v,"datasources"),"datasource",name="review",caption="Gameplay evidence")
    dep=add(v,"datasource-dependencies",datasource="review")
    for col in review_source.findall("column"):dep.append(deepcopy(col))
    for n in detail_fields:add(dep,"column-instance",column=f"[{n}]",name=f"[none:{n}:nk]",derivation="None",pivot="key",type="nominal")
    filter_deck=title in ("Deck routes","Card evidence")
    filters=[("section",title)]+([("deck_label","01 | "+player_name)] if filter_deck else [])
    for k,val in filters:
        fil=add(v,"filter",{"class":"categorical","column":review_field(k)})
        add(fil,"groupfilter",function="member",level=f"[none:{k}:nk]",member=json.dumps(val,ensure_ascii=False))
    add(v,"aggregation",value="true")
    pane=add(add(t,"panes"),"pane");add(add(pane,"view"),"breakdown",value="auto");add(pane,"mark",{"class":"Text"})
    enc=add(pane,"encodings");add(enc,"text",column=review_field("detail"))
    for k in ("cost","conditions","timing","url"):add(enc,"tooltip",column=review_field(k))
    tool=add(add(pane,"customized-tooltip"),"formatted-text")
    add(tool,"run").text="\n".join(review_field(k) for k in ("heading","cost","timing","conditions","detail","url"))
    columns=["deck_label","heading"] if title!="Definitions" else ["heading"]
    if title=="Bracket means":columns=["heading"]
    if title=="Card evidence":columns=["heading","cost","timing"]
    add(t,"rows").text="("+ " / ".join(review_field(k) for k in columns)+")" if len(columns)>1 else review_field(columns[0])
    add(t,"cols").text=review_field("deck_label") if title=="Bracket means" else None
    d=add(boards,"dashboard",name=name);add(d,"style");add(d,"size",{"minwidth":"1000","maxwidth":"1600","minheight":"700","maxheight":"980","sizing-mode":"range"})
    z=add(d,"zones");add(z,"zone",id="1",name=w.get("name"),x="0",y="13000" if filter_deck else "0",w="100000",h="87000" if filter_deck else "100000")
    if filter_deck:add(z,"zone",{"id":"2","name":w.get("name"),"type-v2":"filter","param":review_field("deck_label"),"mode":"checkdropdown","x":"0","y":"0","w":"60000","h":"12000"})
ET.indent(tree,space="  ");tree.write(path,encoding="utf-8",xml_declaration=True)
# A single refresh parameter serves the four detailed Power BI pages.
m["expressions"]=[e for e in m["expressions"] if e["name"] not in ("ReviewFile","ReviewData")]
m["expressions"] += [{**expressions["GameplayFile"],"name":"ReviewFile"},{"name":"ReviewData","kind":"m","expression":read_csv(detail_file,"ReviewFile")}]
m["tables"]=[t for t in m["tables"] if not t["name"].startswith("Evidence | ")]
detail_ids=[]
for i,(title,columns) in enumerate(pages_to_fields.items()):
    table="Evidence | "+title
    schema=[{"name":k,"dataType":"string","sourceColumn":k,"summarizeBy":"none"} for k in detail_fields if k!="deck_order"]
    schema.append({"name":"deck_order","dataType":"int64","sourceColumn":"deck_order","summarizeBy":"none"})
    next(c for c in schema if c["name"]=="deck")["sortByColumn"]="deck_order"
    m["tables"].append({"name":table,"columns":schema,"partitions":[{"name":table,"mode":"import","source":{"type":"m","expression":f'Table.TransformColumnTypes(Table.SelectRows(ReviewData, each [section] = "{title}"), {{{{"deck_order",Int64.Type}}}})'}}]})
    pid=f"ac119f873dc262f2{i:04x}";detail_ids.append(pid)
    writej(pages/pid/"page.json",{**page_format,"name":pid,"displayName":title,"height":850,"width":1500})
    fields=list(columns)
    if title in ("Deck routes","Card evidence"):
        fields=["deck"]+fields
        visual(1,"slicer",(14,8,700,104),state(Values=[("deck",False)]),"Deck")
    visual(2,"tableEx",(14,120 if title in ("Deck routes","Card evidence") else 15,1472,718),
           state(Values=[(k,False) for k in fields]),title)
writej(model_path,model)
index["pageOrder"]=ids+detail_ids+[p for p in index["pageOrder"] if p not in ids+detail_ids]
writej(pages/"pages.json",index)


In [ ]:
# Reuse the single-chart views rather than recomputing measures for the gallery.
gameplay_order=["Printed curve","Deployment routes","Interaction targets","Interaction methods","Recovery routes"]
gallery_order=list(dict.fromkeys(gameplay_order+[v for v in ["Commander identity","Printed card colors","Casting-cost emphasis",
    "Card types","Creature types","Creature composition","Human-adjusted tribes"] if v in views]+views))
row_height=580
height=row_height*len(gallery_order)
gallery_name="Across the decks - all charts"
previous=boards.find(f'dashboard[@name="{gallery_name}"]')
if previous is not None: boards.remove(previous)
gallery=ET.Element("dashboard",name=gallery_name);boards.insert(0,gallery)
add(gallery,"style")
add(gallery,"size",{"minwidth":"1400","maxwidth":"1400","minheight":str(height),"maxheight":str(height),"sizing-mode":"fixed"})
zones=add(gallery,"zones")
for idx,title in enumerate(gallery_order):
    a=round(100000*idx/len(gallery_order));b=round(100000*(idx+1)/len(gallery_order))
    add(zones,"zone",{"id":str(idx+1),"x":"0","y":str(a),"w":"100000","h":str(b-a),"name":"Compare | "+title+" marks"})
ET.indent(tree,space="  ");tree.write(path,encoding="utf-8",xml_declaration=True)
gallery_id="a9c311c85ef04ed1a300"
writej(pages/gallery_id/"page.json",{**page_format,"name":gallery_id,"displayName":"Across the decks | All charts",
       "displayOption":"FitToWidth","width":1500,"height":height})
for idx,title in enumerate(gallery_order):
    page_id=ids[views.index(title)]
    base_visual=next(p for p in (pages/page_id/"visuals").glob("*/*") if p.parent.name.endswith("0001"))
    chart=readj(base_visual)
    visual_id=f"a70ac34b2bce48ad{idx:04x}";chart["name"]=visual_id
    chart["position"]={"x":12,"y":idx*row_height+8,"width":1476,"height":row_height-16,"z":idx,"tabOrder":idx}
    writej(pages/gallery_id/"visuals"/visual_id/"visual.json",chart)
index["pageOrder"]=[gallery_id]+[p for p in index["pageOrder"] if p!=gallery_id]
index["activePageName"]=gallery_id
writej(pages/"pages.json",index)
